# Explainable Transfer Learning for Fabric and Textile Defect Classification

**Student:** Alessandro Ferrão
**Course:** Machine Learning and Smart Systems — University of Europe for Applied Sciences
**Topic 1.4:** Explainable Transfer Learning for Fabric and Textile Defect Classification

This notebook trains and compares **three** architectures on the Ten Fabrics Dataset (TFD):
- **Custom CNN** — trained from scratch (baseline)
- **EfficientNetV2-S** — ImageNet pretrained, fine-tuned
- **ConvNeXt-Tiny** — ImageNet pretrained, fine-tuned

Each model is evaluated with accuracy, balanced accuracy, macro-F1, weighted-F1, and full per-class metrics, and explained using Grad-CAM.

## Before running
1. Click **Add Data** (top-right panel) → search **"Ten Fabrics Dataset TFD"** (by saharshakir) → **Add**.
2. In notebook Settings (right sidebar) → **Accelerator** → select **GPU T4 x2** (or P100).
3. Use **Save Version → Save & Run All (Commit)** to execute the entire notebook end-to-end in the background — this is the version you will download and submit.

If a cell fails: read the error, fix that cell, and re-run **from that cell onward**. You do not need to restart from the top unless the dataset path or an early variable changes.

*Note: SHAP pixel-attribution is intentionally omitted from this notebook to maximise the chance of a clean, uninterrupted full run — Grad-CAM is the primary and more reliable explainability method used here and is fully implemented below.*


In [1]:
# Install packages not preinstalled on Kaggle
!pip install -q gradio


In [2]:
import os
import json
import time
import random
import shutil
import zipfile

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, classification_report, confusion_matrix,
)

# ── Reproducibility ────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected — set Accelerator to GPU in notebook Settings.")


Using device: cuda
GPU: Tesla T4


## 1. Locate the Dataset

Kaggle mounts added datasets under `/kaggle/input/<dataset-slug>/...`, but the internal folder
nesting varies by dataset. The cell below searches automatically for the folder that directly
contains the ten fabric sub-folders (`001` … `010`), so no path needs to be hardcoded.

In [3]:
def find_dataset_root(base="/kaggle/input"):
    """Search for the folder that directly contains sub-folders 001..010."""
    expected = {f"{i:03d}" for i in range(1, 11)}
    for root, dirs, files in os.walk(base):
        if expected.issubset(set(dirs)):
            return root
    return None

DATASET_ROOT = find_dataset_root()

if DATASET_ROOT is None:
    print("Could not auto-detect the dataset. Contents of /kaggle/input:")
    for root, dirs, files in os.walk("/kaggle/input"):
        print(root, "->", dirs[:10])
    raise FileNotFoundError(
        "Add the 'Ten Fabrics Dataset (TFD)' via Add Data (top-right panel), then re-run this cell."
    )

print("Dataset root detected at:", DATASET_ROOT)
print("Contents:", os.listdir(DATASET_ROOT)[:15])


Dataset root detected at: /kaggle/input/datasets/saharshakir/ten-fabrics-dataset-tfd/TFD Textile Dataset
Contents: ['007', '009', '001', '006', '008', '002', '004', '003', '005', '010']


## 2. Configuration

All hyperparameters and paths in one place. `MODELS` now includes **all three** architectures —
ConvNeXtTiny was excluded in the local Mac run because it took ~15 minutes per epoch on Apple MPS;
on a Kaggle GPU this is no longer a constraint.

In [4]:
# ── Paths ──────────────────────────────────────────────────────────
WORK_DIR   = "/kaggle/working"
DATA_DIR   = os.path.join(WORK_DIR, "data", "TFD")     # reorganised 20-class dataset
OUTPUT_DIR = os.path.join(WORK_DIR, "outputs")
MODEL_DIR  = os.path.join(WORK_DIR, "models")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Dataset ────────────────────────────────────────────────────────
IMG_SIZE    = 224
TRAIN_SPLIT = 0.70
VAL_SPLIT   = 0.15
TEST_SPLIT  = 0.15

# ── Training ───────────────────────────────────────────────────────
BATCH_SIZE    = 32
NUM_EPOCHS    = 30
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-4
PATIENCE      = 7

# ── Models to train (all three now that GPU is available) ─────────
MODELS = ["custom_cnn", "efficientnet_v2_s", "convnext_tiny"]

# ── Explainability ─────────────────────────────────────────────────
NUM_EXPLAIN_SAMPLES = 5


## 3. Reorganise the Dataset into 20 Classes

The raw TFD folders (`001`…`010`) each pair a `.png` image with a `.txt` annotation file. Line 5 of
each `.txt` file holds the defect-patch count (0 = clean, >0 = defective). This cell reads that count
and copies each image into one of 20 class folders: `fabricXX_clean` / `fabricXX_defective`.

In [5]:
fabric_names = {f"{i:03d}": f"fabric{i:02d}" for i in range(1, 11)}

moved = 0
for folder in sorted(os.listdir(DATASET_ROOT)):
    folder_path = os.path.join(DATASET_ROOT, folder)
    if not os.path.isdir(folder_path) or folder not in fabric_names:
        continue

    fabric = fabric_names[folder]
    for fname in sorted(os.listdir(folder_path)):
        if not fname.endswith(".txt"):
            continue
        txt_path = os.path.join(folder_path, fname)
        img_name = fname.replace(".txt", ".png")
        img_path = os.path.join(folder_path, img_name)
        if not os.path.exists(img_path):
            continue

        with open(txt_path) as f:
            lines = f.read().strip().split("\n")
        defect_count = int(lines[4].strip())

        label     = "defective" if defect_count > 0 else "clean"
        class_dir = os.path.join(DATA_DIR, f"{fabric}_{label}")
        os.makedirs(class_dir, exist_ok=True)

        shutil.copy2(img_path, os.path.join(class_dir, img_name))
        moved += 1

print(f"Organised {moved} images into {len(os.listdir(DATA_DIR))} classes.\n")
for cls in sorted(os.listdir(DATA_DIR)):
    cls_path = os.path.join(DATA_DIR, cls)
    count = len([f for f in os.listdir(cls_path) if f.endswith(".png")])
    print(f"  {cls}: {count} images")


Organised 2969 images into 20 classes.

  fabric01_clean: 127 images
  fabric01_defective: 27 images
  fabric02_clean: 114 images
  fabric02_defective: 22 images
  fabric03_clean: 95 images
  fabric03_defective: 49 images
  fabric04_clean: 318 images
  fabric04_defective: 78 images
  fabric05_clean: 76 images
  fabric05_defective: 34 images
  fabric06_clean: 205 images
  fabric06_defective: 140 images
  fabric07_clean: 553 images
  fabric07_defective: 207 images
  fabric08_clean: 262 images
  fabric08_defective: 67 images
  fabric09_clean: 302 images
  fabric09_defective: 65 images
  fabric10_clean: 180 images
  fabric10_defective: 48 images


## 4. Dataset Class, Transforms, and Group-Aware Split

Patches sharing the first 6 characters of their filename (e.g. `001-04`) come from the same source
image and are kept together in one partition (train, val, or test) via `StratifiedGroupKFold`. This
prevents texture leakage between partitions.

In [6]:
def get_transforms(split):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    if split == "train":
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])


class TextileDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


def get_dataloaders(data_dir=DATA_DIR):
    class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
    class_to_idx = {c: i for i, c in enumerate(class_names)}

    all_paths, all_labels, all_groups = [], [], []
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}
    for cls in class_names:
        cls_dir = os.path.join(data_dir, cls)
        for fname in sorted(os.listdir(cls_dir)):
            if os.path.splitext(fname)[1].lower() not in valid_exts:
                continue
            all_paths.append(os.path.join(cls_dir, fname))
            all_labels.append(class_to_idx[cls])
            all_groups.append(fname[:6])

    all_paths  = np.array(all_paths)
    all_labels = np.array(all_labels)
    all_groups = np.array(all_groups)

    print(f"[Dataset] {len(all_paths)} images | {len(class_names)} classes")

    def split_once(paths, labels, groups, test_frac):
        n_splits = round(1 / test_frac)
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        for train_idx, test_idx in sgkf.split(paths, labels, groups):
            return train_idx, test_idx

    remaining_idx, test_idx = split_once(all_paths, all_labels, all_groups, TEST_SPLIT)
    val_frac = VAL_SPLIT / (TRAIN_SPLIT + VAL_SPLIT)
    train_idx, val_idx = split_once(
        all_paths[remaining_idx], all_labels[remaining_idx], all_groups[remaining_idx], val_frac
    )
    train_idx = remaining_idx[train_idx]
    val_idx   = remaining_idx[val_idx]

    print(f"[Dataset] train: {len(train_idx)} | val: {len(val_idx)} | test: {len(test_idx)}")

    def make_samples(indices):
        return [(all_paths[i], all_labels[i]) for i in indices]

    train_ds = TextileDataset(make_samples(train_idx), get_transforms("train"))
    val_ds   = TextileDataset(make_samples(val_idx),   get_transforms("val"))
    test_ds  = TextileDataset(make_samples(test_idx),  get_transforms("test"))

    label_counts  = np.bincount(all_labels[train_idx], minlength=len(class_names))
    class_weights = torch.tensor(1.0 / (label_counts + 1e-6), dtype=torch.float32)
    class_weights /= class_weights.sum()

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, val_loader, test_loader, class_names, class_weights


train_loader, val_loader, test_loader, class_names, class_weights = get_dataloaders()
num_classes = len(class_names)

with open(os.path.join(OUTPUT_DIR, "class_names.json"), "w") as f:
    json.dump(class_names, f, indent=2)

print("\nClasses:", class_names)


[Dataset] 2969 images | 20 classes
[Dataset] train: 2139 | val: 400 | test: 430

Classes: ['fabric01_clean', 'fabric01_defective', 'fabric02_clean', 'fabric02_defective', 'fabric03_clean', 'fabric03_defective', 'fabric04_clean', 'fabric04_defective', 'fabric05_clean', 'fabric05_defective', 'fabric06_clean', 'fabric06_defective', 'fabric07_clean', 'fabric07_defective', 'fabric08_clean', 'fabric08_defective', 'fabric09_clean', 'fabric09_defective', 'fabric10_clean', 'fabric10_defective']


## 5. Model Architectures

Three architectures are defined:
1. **CustomCNN** — 5-block CNN, trained from scratch (baseline)
2. **EfficientNetV2-S** — ImageNet pretrained, full fine-tuning
3. **ConvNeXt-Tiny** — ImageNet pretrained, full fine-tuning

In [7]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.1),

            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.1),

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2),

            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2),

            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


class EfficientNetV2(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        weights  = models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
        backbone = models.efficientnet_v2_s(weights=weights)
        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes),
        )
        self.features   = backbone.features
        self.avgpool    = backbone.avgpool
        self.classifier = backbone.classifier

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


class ConvNeXtTiny(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        weights  = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
        backbone = models.convnext_tiny(weights=weights)
        in_features = backbone.classifier[2].in_features
        backbone.classifier[2] = nn.Linear(in_features, num_classes)
        self.features   = backbone.features
        self.avgpool    = backbone.avgpool
        self.classifier = backbone.classifier

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        return self.classifier(x)


def get_model(model_name, num_classes):
    registry = {
        "custom_cnn"        : CustomCNN,
        "efficientnet_v2_s" : EfficientNetV2,
        "convnext_tiny"     : ConvNeXtTiny,
    }
    if model_name not in registry:
        raise ValueError(f"Unknown model '{model_name}'.")
    return registry[model_name](num_classes)


def model_info(model):
    total = sum(p.numel() for p in model.parameters())
    size  = sum(p.nelement() * p.element_size() for p in model.parameters())
    size += sum(b.nelement() * b.element_size() for b in model.buffers())
    return {"parameters": total, "size_mb": size / 1e6}


## 6. Training Function

Weighted cross-entropy loss (per-class weights from Section 4) + AdamW + `ReduceLROnPlateau` +
early stopping on validation macro-F1.

In [8]:
def run_epoch(model, loader, criterion, optimizer, is_train):
    model.train() if is_train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if is_train:
                optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            if is_train:
                loss.backward()
                optimizer.step()

            preds = outputs.argmax(dim=1)
            total_loss += loss.item() * imgs.size(0)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / total
    accuracy = correct / total
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, accuracy, macro_f1


def train_model(model_name):
    checkpoint = os.path.join(MODEL_DIR, f"{model_name}_best.pth")

    model     = get_model(model_name, num_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [],
               "train_f1": [], "val_f1": []}
    best_val_f1, best_epoch, patience_ctr = -1.0, 0, 0

    print(f"\n{'='*55}\n  Training: {model_name.upper()}  |  device: {DEVICE}\n{'='*55}")

    t_start = time.time()
    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_acc, tr_f1 = run_epoch(model, train_loader, criterion, optimizer, True)
        vl_loss, vl_acc, vl_f1 = run_epoch(model, val_loader,   criterion, optimizer, False)
        scheduler.step(vl_f1)

        history["train_loss"].append(tr_loss); history["val_loss"].append(vl_loss)
        history["train_acc"].append(tr_acc);   history["val_acc"].append(vl_acc)
        history["train_f1"].append(tr_f1);     history["val_f1"].append(vl_f1)

        print(f"Epoch {epoch:>3}/{NUM_EPOCHS} | train loss {tr_loss:.4f} acc {tr_acc:.3f} F1 {tr_f1:.3f} "
              f"| val loss {vl_loss:.4f} acc {vl_acc:.3f} F1 {vl_f1:.3f} | {time.time()-t0:.1f}s")

        if vl_f1 > best_val_f1:
            best_val_f1, best_epoch, patience_ctr = vl_f1, epoch, 0
            torch.save(model.state_dict(), checkpoint)
            print(f"  New best val F1 = {best_val_f1:.4f} (saved)")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"  Early stopping at epoch {epoch}.")
                break

    total_time = time.time() - t_start
    print(f"\n  Best val macro-F1 = {best_val_f1:.4f} at epoch {best_epoch}  |  total time {total_time/60:.1f} min")

    del model
    torch.cuda.empty_cache()

    return {"history": history, "best_epoch": best_epoch, "best_val_f1": best_val_f1,
            "total_time_min": round(total_time / 60, 2)}


## 7. Train All Three Models

Results are saved to disk after each model completes, so a Kaggle session interruption does not
lose progress on models already trained.

In [9]:
all_results = {}
for model_name in MODELS:
    result = train_model(model_name)
    all_results[model_name] = result

    with open(os.path.join(OUTPUT_DIR, "training_results.json"), "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"  Results saved after {model_name}\n")

print("Training complete for all models:", list(all_results.keys()))



  Training: CUSTOM_CNN  |  device: cuda
Epoch   1/30 | train loss 2.5024 acc 0.307 F1 0.219 | val loss 1.6466 acc 0.545 F1 0.321 | 15.5s
  New best val F1 = 0.3211 (saved)
Epoch   2/30 | train loss 1.6619 acc 0.461 F1 0.373 | val loss 1.0921 acc 0.440 F1 0.332 | 13.3s
  New best val F1 = 0.3324 (saved)
Epoch   3/30 | train loss 1.3413 acc 0.470 F1 0.394 | val loss 0.9368 acc 0.532 F1 0.318 | 13.3s
Epoch   4/30 | train loss 1.1293 acc 0.470 F1 0.400 | val loss 0.7914 acc 0.598 F1 0.397 | 13.5s
  New best val F1 = 0.3973 (saved)
Epoch   5/30 | train loss 1.0265 acc 0.493 F1 0.435 | val loss 0.7613 acc 0.665 F1 0.380 | 13.8s
Epoch   6/30 | train loss 0.9682 acc 0.505 F1 0.443 | val loss 0.7254 acc 0.610 F1 0.400 | 13.9s
  New best val F1 = 0.4004 (saved)
Epoch   7/30 | train loss 0.9867 acc 0.505 F1 0.449 | val loss 0.7374 acc 0.370 F1 0.276 | 14.2s
Epoch   8/30 | train loss 0.9291 acc 0.506 F1 0.461 | val loss 0.7351 acc 0.725 F1 0.403 | 14.5s
  New best val F1 = 0.4031 (saved)
Epoch   

100%|██████████| 82.7M/82.7M [00:00<00:00, 153MB/s] 



  Training: EFFICIENTNET_V2_S  |  device: cuda
Epoch   1/30 | train loss 1.7549 acc 0.524 F1 0.431 | val loss 0.8626 acc 0.605 F1 0.389 | 29.0s
  New best val F1 = 0.3888 (saved)
Epoch   2/30 | train loss 0.8637 acc 0.586 F1 0.525 | val loss 0.7925 acc 0.645 F1 0.405 | 28.7s
  New best val F1 = 0.4047 (saved)
Epoch   3/30 | train loss 0.8111 acc 0.555 F1 0.485 | val loss 0.7191 acc 0.695 F1 0.518 | 28.6s
  New best val F1 = 0.5181 (saved)
Epoch   4/30 | train loss 0.7444 acc 0.631 F1 0.550 | val loss 0.7046 acc 0.630 F1 0.522 | 28.6s
  New best val F1 = 0.5223 (saved)
Epoch   5/30 | train loss 0.6993 acc 0.680 F1 0.587 | val loss 0.6049 acc 0.733 F1 0.576 | 28.7s
  New best val F1 = 0.5757 (saved)
Epoch   6/30 | train loss 0.6185 acc 0.745 F1 0.651 | val loss 0.5591 acc 0.845 F1 0.702 | 28.5s
  New best val F1 = 0.7019 (saved)
Epoch   7/30 | train loss 0.5954 acc 0.763 F1 0.663 | val loss 0.5155 acc 0.833 F1 0.704 | 28.4s
  New best val F1 = 0.7044 (saved)
Epoch   8/30 | train loss 0.

100%|██████████| 109M/109M [00:00<00:00, 196MB/s]  



  Training: CONVNEXT_TINY  |  device: cuda
Epoch   1/30 | train loss 1.2784 acc 0.505 F1 0.450 | val loss 0.7406 acc 0.583 F1 0.489 | 51.2s
  New best val F1 = 0.4893 (saved)
Epoch   2/30 | train loss 0.7832 acc 0.558 F1 0.497 | val loss 0.7038 acc 0.583 F1 0.492 | 41.8s
  New best val F1 = 0.4920 (saved)
Epoch   3/30 | train loss 0.6949 acc 0.660 F1 0.592 | val loss 0.7588 acc 0.670 F1 0.555 | 41.9s
  New best val F1 = 0.5546 (saved)
Epoch   4/30 | train loss 0.6209 acc 0.735 F1 0.638 | val loss 0.5423 acc 0.748 F1 0.661 | 41.7s
  New best val F1 = 0.6608 (saved)
Epoch   5/30 | train loss 0.5654 acc 0.773 F1 0.685 | val loss 0.5438 acc 0.848 F1 0.692 | 41.7s
  New best val F1 = 0.6925 (saved)
Epoch   6/30 | train loss 0.5267 acc 0.828 F1 0.722 | val loss 0.4290 acc 0.892 F1 0.776 | 41.8s
  New best val F1 = 0.7760 (saved)
Epoch   7/30 | train loss 0.5080 acc 0.840 F1 0.736 | val loss 0.5069 acc 0.828 F1 0.732 | 41.5s
Epoch   8/30 | train loss 0.5407 acc 0.818 F1 0.707 | val loss 0.39

## 8. Evaluation on Held-Out Test Set

Computes accuracy, balanced accuracy, macro-F1, weighted-F1, per-class precision/recall/F1,
confusion matrix, inference latency, parameter count, and model size for every trained model.

In [10]:
def evaluate_on_test(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(DEVICE)
            out  = model(imgs)
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(lbls.numpy())
    return np.array(preds), np.array(labels)


def measure_inference_time(model, n_runs=100):
    dummy = torch.randn(1, 3, 224, 224).to(DEVICE)
    model.eval()
    with torch.no_grad():
        for _ in range(10):
            model(dummy)
        t0 = time.perf_counter()
        for _ in range(n_runs):
            model(dummy)
    return (time.perf_counter() - t0) / n_runs * 1000


def plot_confusion_matrix(cm, model_name):
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix — {model_name}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, f"confusion_matrix_{model_name}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Confusion matrix saved -> {path}")


def plot_training_curves(history, model_name):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, history["train_loss"], label="Train")
    axes[0].plot(epochs, history["val_loss"], label="Val")
    axes[0].set_title(f"{model_name} — Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[1].plot(epochs, history["train_f1"], label="Train")
    axes[1].plot(epochs, history["val_f1"], label="Val")
    axes[1].set_title(f"{model_name} — Macro-F1"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, f"training_curves_{model_name}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Training curves saved -> {path}")


with open(os.path.join(OUTPUT_DIR, "training_results.json")) as f:
    training_results = json.load(f)

evaluation_report = {}

for model_name in MODELS:
    print(f"\n{'-'*55}\n  Evaluating: {model_name.upper()}\n{'-'*55}")

    checkpoint = os.path.join(MODEL_DIR, f"{model_name}_best.pth")
    if not os.path.exists(checkpoint):
        print("  No checkpoint found -- skipping.")
        continue

    model = get_model(model_name, num_classes).to(DEVICE)
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE))

    preds, labels = evaluate_on_test(model, test_loader)

    acc      = accuracy_score(labels, preds)
    bal_acc  = balanced_accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    wt_f1    = f1_score(labels, preds, average="weighted", zero_division=0)
    cm_arr   = confusion_matrix(labels, preds)
    infer_ms = measure_inference_time(model)
    info     = model_info(model)

    evaluation_report[model_name] = {
        "accuracy": round(float(acc), 4),
        "balanced_accuracy": round(float(bal_acc), 4),
        "macro_f1": round(float(macro_f1), 4),
        "weighted_f1": round(float(wt_f1), 4),
        "inference_ms": round(infer_ms, 2),
        "parameters": info["parameters"],
        "size_mb": round(info["size_mb"], 2),
    }

    print(f"  Accuracy: {acc:.4f} | Balanced Acc: {bal_acc:.4f} | Macro-F1: {macro_f1:.4f} | Weighted-F1: {wt_f1:.4f}")
    print(f"  Inference: {infer_ms:.2f} ms | Params: {info['parameters']:,} | Size: {info['size_mb']:.2f} MB")
    print(classification_report(labels, preds, target_names=class_names, zero_division=0))

    plot_confusion_matrix(cm_arr, model_name)
    if model_name in training_results:
        plot_training_curves(training_results[model_name]["history"], model_name)

    del model
    torch.cuda.empty_cache()

with open(os.path.join(OUTPUT_DIR, "evaluation_report.json"), "w") as f:
    json.dump(evaluation_report, f, indent=2)

print("\nEvaluation complete for all models. Report saved to outputs/evaluation_report.json")



-------------------------------------------------------
  Evaluating: CUSTOM_CNN
-------------------------------------------------------
  Accuracy: 0.6628 | Balanced Acc: 0.5012 | Macro-F1: 0.4196 | Weighted-F1: 0.5707
  Inference: 1.73 ms | Params: 2,491,828 | Size: 9.98 MB
                    precision    recall  f1-score   support

    fabric01_clean       0.85      1.00      0.92        17
fabric01_defective       0.00      0.00      0.00         3
    fabric02_clean       0.85      1.00      0.92        17
fabric02_defective       0.00      0.00      0.00         3
    fabric03_clean       0.00      0.00      0.00         3
fabric03_defective       0.70      1.00      0.82         7
    fabric04_clean       0.84      1.00      0.91        42
fabric04_defective       0.00      0.00      0.00         8
    fabric05_clean       0.55      1.00      0.71        11
fabric05_defective       0.00      0.00      0.00         9
    fabric06_clean       0.38      1.00      0.55        19
f

## 9. Grad-CAM Explainability

Generates heatmaps for 5 defective test images per model, hooking into the last convolutional
block of each architecture.

In [11]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self._fwd = target_layer.register_forward_hook(self._save_act)
        self._bwd = target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, module, input, output):
        self.activations = output.detach()

    def _save_grad(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, img_tensor, class_idx=None):
        self.model.eval()
        self.model.zero_grad()
        output = self.model(img_tensor)
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()
        output[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=img_tensor.shape[2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam, class_idx

    def remove(self):
        self._fwd.remove()
        self._bwd.remove()


def get_target_layer(model, model_name):
    if model_name == "custom_cnn":
        for layer in reversed(list(model.features.children())):
            if isinstance(layer, torch.nn.Conv2d):
                return layer
    elif model_name in ("efficientnet_v2_s", "convnext_tiny"):
        return model.features[-1]
    raise ValueError(f"Cannot find target layer for {model_name}")


def denorm(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = tensor.cpu().permute(1, 2, 0).numpy()
    img  = (img * std + mean) * 255
    return np.clip(img, 0, 255).astype(np.uint8)


def overlay(img_np, heatmap, alpha=0.5):
    colormap  = cm.get_cmap("jet")
    heatmap_c = colormap(heatmap)[..., :3]
    blended   = alpha * heatmap_c + (1 - alpha) * img_np / 255.0
    return np.clip(blended, 0, 1)


def save_gradcam_grid(samples, heatmaps, preds, model_name):
    n = len(samples)
    fig, axes = plt.subplots(n, 2, figsize=(8, 4 * n))
    if n == 1:
        axes = [axes]
    for i, ((img_t, true_label), pred, hmap) in enumerate(zip(samples, preds, heatmaps)):
        img_np  = denorm(img_t)
        blended = overlay(img_np, hmap)
        axes[i][0].imshow(img_np)
        axes[i][0].set_title(f"True: {class_names[true_label]}\nPred: {class_names[pred]}", fontsize=8)
        axes[i][0].axis("off")
        axes[i][1].imshow(blended)
        axes[i][1].set_title("Grad-CAM", fontsize=8)
        axes[i][1].axis("off")
    plt.suptitle(f"Grad-CAM — {model_name}", fontsize=12)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "explanations", f"gradcam_{model_name}.png")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Grad-CAM saved -> {path}")


defective_indices = [i for i, name in enumerate(class_names) if "defective" in name]

for model_name in MODELS:
    print(f"\nGenerating Grad-CAM for {model_name}...")
    checkpoint = os.path.join(MODEL_DIR, f"{model_name}_best.pth")
    if not os.path.exists(checkpoint):
        print("  No checkpoint -- skipping.")
        continue

    model = get_model(model_name, num_classes).to(DEVICE)
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE))
    model.eval()

    samples, heatmaps, preds = [], [], []
    target_layer = get_target_layer(model, model_name)
    gcam = GradCAM(model, target_layer)

    for imgs, labels in test_loader:
        for i in range(len(imgs)):
            if len(samples) >= NUM_EXPLAIN_SAMPLES:
                break
            if labels[i].item() not in defective_indices:
                continue
            img_t = imgs[i:i+1].to(DEVICE)
            img_t.requires_grad_(True)
            with torch.enable_grad():
                hmap, pred = gcam.generate(img_t)
            samples.append((imgs[i], labels[i].item()))
            heatmaps.append(hmap)
            preds.append(pred)
        if len(samples) >= NUM_EXPLAIN_SAMPLES:
            break

    gcam.remove()
    save_gradcam_grid(samples, heatmaps, preds, model_name)

    del model
    torch.cuda.empty_cache()

print("\nGrad-CAM complete for all models.")



Generating Grad-CAM for custom_cnn...


/tmp/ipykernel_58/2429052873.py:54: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colormap  = cm.get_cmap("jet")


  Grad-CAM saved -> /kaggle/working/outputs/explanations/gradcam_custom_cnn.png

Generating Grad-CAM for efficientnet_v2_s...
  Grad-CAM saved -> /kaggle/working/outputs/explanations/gradcam_efficientnet_v2_s.png

Generating Grad-CAM for convnext_tiny...
  Grad-CAM saved -> /kaggle/working/outputs/explanations/gradcam_convnext_tiny.png

Grad-CAM complete for all models.


## 10. Package Results for Download

Zips the `outputs/` (metrics, figures, Grad-CAM) and `models/` (checkpoints) folders separately so
they stay under Kaggle's per-file size comfort zone.

In [12]:
def zip_dir(path, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, os.path.dirname(path))
                zf.write(file_path, arcname)

zip_dir(OUTPUT_DIR, os.path.join(WORK_DIR, "outputs.zip"))
zip_dir(MODEL_DIR,  os.path.join(WORK_DIR, "models.zip"))

print("Zipped outputs.zip and models.zip -- download both from the Output tab on the right.")


Zipped outputs.zip and models.zip -- download both from the Output tab on the right.


## Next Steps

1. Use **Save Version → Save & Run All (Commit)** if you have not already — this executes every
   cell top to bottom in the background and is the version you submit.
2. Once the commit finishes, open that saved version and download **`outputs.zip`** and
   **`models.zip`** from the **Output** tab.
3. Download the notebook itself with its outputs: **File → Download → Download .ipynb**.
4. Upload the executed `.ipynb`, plus the unzipped contents of `outputs/`, to your GitHub
   repository (see the `Codes/` and `Outputs/` folders discussed earlier).
5. Update your local `app.py` Gradio demo to use the new checkpoints from `models.zip` — the
   class list may be identical, but re-copy the checkpoints so the demo reflects the Kaggle-trained
   weights (and now also offers ConvNeXt-Tiny as a model option).
